Question 1

In [ ]:
import numpy as np

# Camera intrinsics
f = 480
cx, cy = 320, 270
K = np.array([
    [f, 0, cx],
    [0, f, cy],
    [0, 0, 1]
])

# Pose parameters (Global to Camera)
R_G_C = np.array([
    [0.5363, -0.8440, 0],
    [0.8440, 0.5363, 0],
    [0, 0, 1]
])
t_C_to_G = np.array([[-451.2459], [257.0322], [400]])
X_G = np.array([[350], [-250], [-35]])

# (a) Projection Matrix P = K * [R_G_C | -R_G_C * t_C_to_G]
t_cam = -R_G_C @ t_C_to_G
Rt = np.hstack((R_G_C, t_cam))
P = K @ Rt

print("(a) Camera Projection Matrix P:")
print(np.round(P, 4))

# (b) Project 3D point
X_G_homog = np.vstack((X_G, [[1]]))
x_img_homog = P @ X_G_homog

# Normalize by Z
u = x_img_homog[0, 0] / x_img_homog[2, 0]
v = x_img_homog[1, 0] / x_img_homog[2, 0]

print(f"\n(b) Projected Image Coordinates (u, v): ({u:.4f}, {v:.4f})")
print(f"Note: Z is {x_img_homog[2, 0]:.4f}. Because Z is negative, the point is actually behind the camera!")

# (c) Re-projection Error
obs_u, obs_v = 241.5, 169.0
error = np.sqrt((u - obs_u)**2 + (v - obs_v)**2)
print(f"\n(c) Re-projection Error: {error:.4f} pixels")

Question 2

In [ ]:
import cv2
import numpy as np
import glob
import matplotlib.pyplot as plt

# Set this to the number of INNER corners of your physical printed checkerboard.
CHECKERBOARD = (6, 9)
criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001)

# Prepare 3D points in real world space
objp = np.zeros((CHECKERBOARD[0] * CHECKERBOARD[1], 3), np.float32)
objp[:, :2] = np.mgrid[0:CHECKERBOARD[0], 0:CHECKERBOARD[1]].T.reshape(-1, 2)

objpoints = [] # 3d points in real world space
imgpoints = [] # 2d points in image plane

# Ensure this path matches where you put your physical photos
images = glob.glob('calibration_images/*')
print(f"Found {len(images)} images for calibration.")

gray_shape = None
img_snapshot = None

for fname in images:
    img = cv2.imread(fname)
    if img is None: continue

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    gray_shape = gray.shape[::-1]

    # Find the chess board corners
    ret, corners = cv2.findChessboardCorners(gray, CHECKERBOARD, None)

    if ret:
        objpoints.append(objp)
        corners2 = cv2.cornerSubPix(gray, corners, (11, 11), (-1, -1), criteria)
        imgpoints.append(corners2)

        # Save one snapshot for requirement (c) using matplotlib instead of cv2.imshow
        if img_snapshot is None:
            img_snapshot = img.copy()
            cv2.drawChessboardCorners(img_snapshot, CHECKERBOARD, corners2, ret)
            img_snapshot = cv2.cvtColor(img_snapshot, cv2.COLOR_BGR2RGB)

# (c) Submit snapshots of the process
if img_snapshot is not None:
    plt.figure(figsize=(8, 6))
    plt.imshow(img_snapshot)
    plt.title("Calibration Snapshot")
    plt.axis('off')
    plt.show()

# (d) Camera calibration matrix K
if len(objpoints) > 0:
    ret, mtx, dist, rvecs, tvecs = cv2.calibrateCamera(objpoints, imgpoints, gray_shape, None, None)
    print("\n(d) Estimated Camera Calibration Matrix K:")
    print(np.round(mtx, 2))
    print(f"Focal Length (fx, fy): ({mtx[0,0]:.2f}, {mtx[1,1]:.2f})")
    print(f"Principal Point (cx, cy): ({mtx[0,2]:.2f}, {mtx[1,2]:.2f})")
else:
    print("\nError: Could not find checkerboard corners. Check your images and CHECKERBOARD dimensions.")

--- Part 1: Projection of a 3D point ---

(a) Projection Matrix P:
[[ 2.57424000e+02 -4.05120000e+02  3.20000000e+02  9.22904094e+04]
 [ 4.05120000e+02  2.57424000e+02  2.70000000e+02  8.64248200e+03]
 [ 0.00000000e+00  0.00000000e+00  1.00000000e+00 -4.00000000e+02]] 

(b) Projected Image Coordinates: (u, v) = (-626.3651, -176.1574)

(c) Re-projection Error: 933.9826 pixels

--- Part 2: Calibrate your own camera (Framework) ---

--- Part 3: SIFT Extraction and Matching (Framework) ---



Question 3

In [7]:
import cv2
import matplotlib.pyplot as plt

def extract_and_match_sift(img1_path, img2_path):
    img1 = cv2.imread(img1_path, cv2.IMREAD_GRAYSCALE)
    img2 = cv2.imread(img2_path, cv2.IMREAD_GRAYSCALE)

    if img1 is None or img2 is None:
        print("Error: Could not load images. Update your file paths.")
        return

    # (b) Extract SIFT features
    sift = cv2.SIFT_create()
    kp1, des1 = sift.detectAndCompute(img1, None)
    kp2, des2 = sift.detectAndCompute(img2, None)

    # Indicate feature scale and orientation for a representative feature
    if len(kp1) > 0:
        rep_kp = kp1[0]
        print(f"Representative feature (Img 1) -> Scale/Size: {rep_kp.size:.2f}, Orientation: {rep_kp.angle:.2f} degrees\n")

    # (c) Calculate putative matches
    bf = cv2.BFMatcher()
    matches = bf.knnMatch(des1, des2, k=2)

    # Apply Lowe's ratio test to separate inliers from outliers
    good_matches = []
    outliers = []
    for m, n in matches:
        if m.distance < 0.75 * n.distance:
            good_matches.append([m])
        else:
            outliers.append([m])

    print(f"Total Matches: {len(matches)}")
    print(f"Inliers (passed ratio test): {len(good_matches)}")
    print(f"Outliers (failed ratio test): {len(outliers)}\n")

    # (c) Show matches, indicate representative inlier and outlier matches
    fig, axes = plt.subplots(2, 1, figsize=(12, 10))

    # Draw top 20 Inliers
    img_inliers = cv2.drawMatchesKnn(img1, kp1, img2, kp2, good_matches[:20], None,
                                     flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS, matchColor=(0, 255, 0))
    axes[0].imshow(cv2.cvtColor(img_inliers, cv2.COLOR_BGR2RGB))
    axes[0].set_title("Representative Inliers (Top 20, Green)")
    axes[0].axis('off')

    # Draw top 20 Outliers
    img_outliers = cv2.drawMatchesKnn(img1, kp1, img2, kp2, outliers[:20], None,
                                      flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS, matchColor=(255, 0, 0))
    axes[1].imshow(cv2.cvtColor(img_outliers, cv2.COLOR_BGR2RGB))
    axes[1].set_title("Representative Outliers (Top 20, Red)")
    axes[1].axis('off')

    plt.tight_layout()
    plt.show()

# Replace these with the paths to the 2 images of yourself
# extract_and_match_sift('my_photo_1.jpg', 'my_photo_2.jpg')

(a) Camera Projection Matrix P:
[[ 2.57424000e+02 -4.05120000e+02  3.20000000e+02  9.22904094e+04]
 [ 4.05120000e+02  2.57424000e+02  2.70000000e+02  8.64248200e+03]
 [ 0.00000000e+00  0.00000000e+00  1.00000000e+00 -4.00000000e+02]]

(b) Projected Image Coordinates (u, v): (-626.3651, -176.1574)

(c) Re-projection Error: 933.9826 pixels
